# Build Model

In [2]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from datetime import datetime as dt
import bentoml
from joblib import load
import os


pd.set_option("display.max_columns", 100)
pd.set_option('display.max_rows', 50)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
from dota_oracle_common.postgresql import DatabaseEngineFactory
from sqlalchemy.ext.asyncio import AsyncSession

engine = DatabaseEngineFactory.get_engine()
engine

In [ ]:
from dota_oracle_common.repositories.match_repository import MatchRepository


async with AsyncSession(engine) as session:
    match_repo = MatchRepository(session)
    
    matches = await match_repo.get_match_details(
        relationship_fields=[
            "outcome", "team_features", "player_hero_features", "hero_features"
        ]
    )
    
    print(f"number of matches: {len(matches)}")
    



dota_oracle_common.repositories.base_repository - Retrieved 96821 records for MatchTable
dota_oracle_common.repositories.match_repository - Found 96821 MatchTable details.


number of matches: 96821


In [21]:
match_outcome_list = []
hero_features_list = []
player_hero_team_features_list = []

for match in matches:
    outcome = match.outcome
    match_outcome_list.append(outcome.model_dump())
    
    team_features = match.team_features
    hero_features = match.hero_features
    player_hero_features = match.player_hero_features
    
    player_hero_team_features_dict = {**team_features.model_dump(), **player_hero_features.model_dump()}
    
    hero_features_list.append(hero_features.model_dump())
    player_hero_team_features_list.append(player_hero_team_features_dict)
    

print(f"count match_outcome_list: {len(match_outcome_list)}")
print(f"count features_list: {len(player_hero_team_features_list)}")

count match_outcome_list: 96821
count features_list: 96821


In [22]:
outcome_df = pd.DataFrame(match_outcome_list)
player_hero_team_df = pd.DataFrame(player_hero_team_features_list)
hero_df = pd.DataFrame(hero_features_list)

In [23]:
outcome_df

,match_id,radiant_win
0,8230722475,True
1,8230701740,False
2,8230693148,True
3,8230677659,True
4,8230656847,True
...,...,...
96816,5999283181,False
96817,5999249937,False
96818,5999214195,False
96819,5999201501,True


In [24]:
player_hero_team_df

,dire_win_rate,radiant_win_rate,match_id,radiant_dire_matchup,player_hero_3_win_rate,player_hero_0_win_rate,player_hero_128_win_rate,player_hero_130_win_rate,player_hero_132_win_rate,player_hero_1_win_rate,player_hero_2_win_rate,player_hero_4_win_rate,player_hero_129_win_rate,player_hero_131_win_rate
0,0.7,0.5,8230722475,0.333333,0.75,0.600000,0.800000,0.714286,0.80,0.550000,0.600000,0.4,0.769231,0.500000
1,0.5,0.7,8230701740,0.600000,0.70,0.642857,0.300000,0.300000,0.50,0.500000,0.384615,0.4,0.666667,0.550000
2,0.6,0.6,8230693148,0.625000,1.00,0.777778,0.588235,0.600000,0.50,0.714286,0.692308,0.5,0.500000,0.833333
3,0.8,0.4,8230677659,0.350000,0.45,0.250000,0.625000,1.000000,0.00,0.350000,0.625000,0.6,0.600000,1.000000
4,0.4,0.7,8230656847,0.600000,0.40,0.450000,0.384615,0.500000,0.65,0.538462,0.400000,0.5,0.350000,0.600000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96816,0.5,0.5,5999283181,0.500000,0.50,0.500000,0.500000,0.500000,0.50,0.500000,0.500000,0.5,0.500000,0.500000
96817,0.0,1.0,5999249937,1.000000,1.00,0.500000,0.500000,0.500000,0.50,0.500000,0.500000,0.5,0.000000,0.500000
96818,0.5,0.5,5999214195,0.500000,0.50,0.500000,0.500000,0.500000,0.50,0.500000,0.500000,0.5,0.500000,0.500000
96819,0.5,0.5,5999201501,0.500000,0.50,0.500000,0.500000,0.500000,0.50,0.500000,0.500000,0.5,0.500000,0.500000


In [25]:
hero_df

,hero_picks,match_id
0,"[Sven, Tiny, Clockwerk, Shadow Demon, Dark See...",8230722475
1,"[Ancient Apparition, Magnus, Pudge, Puck, Sven...",8230701740
2,"[Tinker, Pangolier, Tiny, Leshrac, Lifestealer...",8230693148
3,"[Zeus, Pudge, Wraith King, Sniper, Death Proph...",8230677659
4,"[Jakiro, Tidehunter, Lina, Rubick, Tiny, Anti-...",8230656847
...,...,...
96816,"[Enchantress, Timbersaw, Tiny, Wraith King, Or...",5999283181
96817,"[Void Spirit, Brewmaster, Terrorblade, Snapfir...",5999249937
96818,"[Centaur Warrunner, Hoodwink, Razor, Grimstrok...",5999214195
96819,"[Ancient Apparition, Enchantress, Magnus, Embe...",5999201501


In [26]:
from dota_oracle_pipeline.feature_transformation.feature_encoder import FeatureEncoder
from dota_oracle_common.repositories.heroes_repository import HeroesRepository

async with AsyncSession(engine) as session:
    heros_repo = HeroesRepository(session)
    hero_map = await heros_repo.get_hero_id_map()

encoded_hero_features = FeatureEncoder.encode_hero_features(hero_features=hero_df, hero_map=hero_map)

In [27]:
encoded_hero_features

,Pugna,Anti-Mage,Axe,Bane,Bloodseeker,Crystal Maiden,Drow Ranger,Earthshaker,Juggernaut,Mirana,Morphling,Shadow Fiend,Phantom Lancer,Puck,Pudge,Razor,Sand King,Storm Spirit,Sven,Tiny,Vengeful Spirit,Windranger,Zeus,Kunkka,Lina,Lion,Shadow Shaman,Slardar,Tidehunter,Witch Doctor,Lich,Riki,Enigma,Tinker,Sniper,Necrophos,Warlock,Beastmaster,Queen of Pain,Venomancer,Faceless Void,Wraith King,Death Prophet,Phantom Assassin,Templar Assassin,Viper,Luna,Dragon Knight,Dazzle,Clockwerk,...,Shadow Demon,Lone Druid,Chaos Knight,Meepo,Treant Protector,Ogre Magi,Undying,Rubick,Disruptor,Nyx Assassin,Naga Siren,Keeper of the Light,Io,Visage,Slark,Medusa,Troll Warlord,Centaur Warrunner,Magnus,Timbersaw,Bristleback,Tusk,Skywrath Mage,Abaddon,Elder Titan,Legion Commander,Techies,Ember Spirit,Earth Spirit,Underlord,Terrorblade,Phoenix,Oracle,Winter Wyvern,Arc Warden,Monkey King,Dark Willow,Pangolier,Grimstroke,Hoodwink,Void Spirit,Snapfire,Mars,Ring Master,Dawnbreaker,Marci,Primal Beast,Muerta,Kez,match_id
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,...,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,8230722475
1,0,0,0,0,0,0,0,0,0,0,0,1,0,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,8230701740
2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,8230693148
3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,8230677659
4,0,1,0,0,0,1,0,0,0,0,0,1,0,0,1,0,1,0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,8230656847
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96816,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,5999283181
96817,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,1,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,5999249937
96818,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,0,0,0,0,0,5999214195
96819,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,5999201501


In [40]:
merged_df = pd.merge(outcome_df,player_hero_team_df, how='inner')
merged_df

,match_id,radiant_win,dire_win_rate,radiant_win_rate,radiant_dire_matchup,player_hero_3_win_rate,player_hero_0_win_rate,player_hero_128_win_rate,player_hero_130_win_rate,player_hero_132_win_rate,player_hero_1_win_rate,player_hero_2_win_rate,player_hero_4_win_rate,player_hero_129_win_rate,player_hero_131_win_rate
0,8230722475,True,0.7,0.5,0.333333,0.75,0.600000,0.800000,0.714286,0.80,0.550000,0.600000,0.4,0.769231,0.500000
1,8230701740,False,0.5,0.7,0.600000,0.70,0.642857,0.300000,0.300000,0.50,0.500000,0.384615,0.4,0.666667,0.550000
2,8230693148,True,0.6,0.6,0.625000,1.00,0.777778,0.588235,0.600000,0.50,0.714286,0.692308,0.5,0.500000,0.833333
3,8230677659,True,0.8,0.4,0.350000,0.45,0.250000,0.625000,1.000000,0.00,0.350000,0.625000,0.6,0.600000,1.000000
4,8230656847,True,0.4,0.7,0.600000,0.40,0.450000,0.384615,0.500000,0.65,0.538462,0.400000,0.5,0.350000,0.600000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96816,5999283181,False,0.5,0.5,0.500000,0.50,0.500000,0.500000,0.500000,0.50,0.500000,0.500000,0.5,0.500000,0.500000
96817,5999249937,False,0.0,1.0,1.000000,1.00,0.500000,0.500000,0.500000,0.50,0.500000,0.500000,0.5,0.000000,0.500000
96818,5999214195,False,0.5,0.5,0.500000,0.50,0.500000,0.500000,0.500000,0.50,0.500000,0.500000,0.5,0.500000,0.500000
96819,5999201501,True,0.5,0.5,0.500000,0.50,0.500000,0.500000,0.500000,0.50,0.500000,0.500000,0.5,0.500000,0.500000


In [41]:
df_final = merged_df.drop('match_id', axis=1)

In [42]:
# Train Test Split
X = df_final.drop('radiant_win', axis=1)
y = df_final['radiant_win']

n_total = len(df_final)
n_test = int(n_total * 0.3)

# Top 30% as test, remaining 70% as train
X_test = X.iloc[:n_test]     # First 30%
X_train = X.iloc[n_test:]    # Remaining 70%
y_test = y.iloc[:n_test]     # First 30%
y_train = y.iloc[n_test:]    # Remaining 70

In [43]:
# Import candidate classifiers

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import lightgbm as lgb

# import metrics

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [44]:
# Initialize models
models = {
    'Logistic Regression': LogisticRegression(
        random_state=42,
        max_iter=1000  # Increase if convergence issues
    ),
    
    'Random Forest': RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1  # Use all cores
    ),
    
    'XGBoost': xgb.XGBClassifier(
        random_state=42,
        eval_metric='logloss',  # Suppress warning
        n_estimators=100
    ),
    
    'LightGBM': lgb.LGBMClassifier(
        random_state=42,
        verbosity=-1,  # Suppress output
        n_estimators=100
    )
}

In [45]:
results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    
    # Train the model
    model.fit(X_train, y_train)
    
    # Make predictions
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]  # For ROC-AUC
    
    # Calculate accuracy
    accuracy = accuracy_score(y_test, y_pred)
    
    # Store results
    results[name] = {
        'model': model,
        'accuracy': accuracy,
        'predictions': y_pred,
        'probabilities': y_pred_proba
    }
    
    print(f"{name} Accuracy: {accuracy:.3f} ({accuracy*100:.1f}%)")


Training Logistic Regression...
Logistic Regression Accuracy: 0.557 (55.7%)

Training Random Forest...
Random Forest Accuracy: 0.547 (54.7%)

Training XGBoost...
XGBoost Accuracy: 0.547 (54.7%)

Training LightGBM...
LightGBM Accuracy: 0.557 (55.7%)


In [46]:
import numpy as np

print("Feature-target correlation:")
for col in X_train.select_dtypes(include=[np.number]).columns:
    corr = X_train[col].corr(y_train)
    print(f"{col}: {corr:.4f}")

Feature-target correlation:
dire_win_rate: -0.1128
radiant_win_rate: 0.1152
radiant_dire_matchup: 0.1547
player_hero_3_win_rate: 0.0577
player_hero_0_win_rate: 0.0432
player_hero_128_win_rate: -0.0566
player_hero_130_win_rate: -0.0561
player_hero_132_win_rate: -0.0573
player_hero_1_win_rate: 0.0482
player_hero_2_win_rate: 0.0525
player_hero_4_win_rate: 0.0513
player_hero_129_win_rate: -0.0538
player_hero_131_win_rate: -0.0575


In [67]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

def create_aggregated_player_features(df):
    """Create aggregated team-level features from individual player win rates"""
    
    print("Creating aggregated player features...")
    
    df = df.copy()
    
    # Define player slots for each team
    radiant_cols = [f'player_hero_{i}_win_rate' for i in range(5)]
    dire_cols = [f'player_hero_{i}_win_rate' for i in range(128, 133)]
    
    # Check which columns exist
    available_radiant = [col for col in radiant_cols if col in df.columns]
    available_dire = [col for col in dire_cols if col in df.columns]
    
    print(f"Available Radiant features: {len(available_radiant)}")
    print(f"Available Dire features: {len(available_dire)}")
    
    if len(available_radiant) == 5 and len(available_dire) == 5:
        # Basic aggregations
        df['radiant_avg_winrate'] = df[available_radiant].mean(axis=1)
        df['dire_avg_winrate'] = df[available_dire].mean(axis=1)
        df['team_winrate_advantage'] = df['radiant_avg_winrate'] - df['dire_avg_winrate']
        
        # Advanced aggregations
        df['radiant_min_winrate'] = df[available_radiant].min(axis=1)
        df['dire_min_winrate'] = df[available_dire].min(axis=1)
        df['weak_link_advantage'] = df['radiant_min_winrate'] - df['dire_min_winrate']
        
        df['radiant_max_winrate'] = df[available_radiant].max(axis=1)
        df['dire_max_winrate'] = df[available_dire].max(axis=1)
        df['star_player_advantage'] = df['radiant_max_winrate'] - df['dire_max_winrate']
        
        df['radiant_consistency'] = df[available_radiant].std(axis=1)
        df['dire_consistency'] = df[available_dire].std(axis=1)
        df['consistency_advantage'] = df['dire_consistency'] - df['radiant_consistency']
        
        # Percentile features
        df['radiant_75th_percentile'] = df[available_radiant].quantile(0.75, axis=1)
        df['dire_75th_percentile'] = df[available_dire].quantile(0.75, axis=1)
        df['top_players_advantage'] = df['radiant_75th_percentile'] - df['dire_75th_percentile']
        
        # Combined features
        if 'radiant_dire_matchup' in df.columns:
            df['total_advantage'] = (
                0.6 * df['radiant_dire_matchup'] + 
                0.4 * df['team_winrate_advantage']
            )
        else:
            df['total_advantage'] = df['team_winrate_advantage']
        
        print("✅ Aggregated features created successfully")
        
    else:
        print("❌ Missing player_hero win rate columns")
        return df
    
    return df

def create_team_aware_hero_features(hero_df):
    """Convert hero_picks to team-aware features"""
    
    print("Creating team-aware hero features...")
    
    # Convert hero_picks to team-aware format
    team_aware_data = []
    
    for _, row in hero_df.iterrows():
        match_id = row['match_id']
        hero_picks = row['hero_picks']
        
        if isinstance(hero_picks, list) and len(hero_picks) == 10:
            # Assume first 5 are Radiant, last 5 are Dire
            radiant_heroes = hero_picks[:5]
            dire_heroes = hero_picks[5:10]
            
            team_aware_data.append({
                'match_id': match_id,
                'radiant_heroes': radiant_heroes,
                'dire_heroes': dire_heroes
            })
    
    team_aware_df = pd.DataFrame(team_aware_data)
    print(f"Converted {len(team_aware_df)} matches to team-aware format")
    
    # Get all unique heroes
    all_heroes = set()
    for _, row in team_aware_df.iterrows():
        all_heroes.update(row['radiant_heroes'])
        all_heroes.update(row['dire_heroes'])
    
    all_heroes = sorted(list(all_heroes))
    print(f"Found {len(all_heroes)} unique heroes")
    
    # Create binary encodings for each team
    radiant_mlb = MultiLabelBinarizer(classes=all_heroes)
    dire_mlb = MultiLabelBinarizer(classes=all_heroes)
    
    radiant_matrix = radiant_mlb.fit_transform(team_aware_df['radiant_heroes'])
    dire_matrix = dire_mlb.fit_transform(team_aware_df['dire_heroes'])
    
    # Create feature dataframes
    radiant_features = pd.DataFrame(
        radiant_matrix,
        columns=[f'radiant_{hero}' for hero in all_heroes],
        index=team_aware_df.index
    )
    
    dire_features = pd.DataFrame(
        dire_matrix,
        columns=[f'dire_{hero}' for hero in all_heroes],
        index=team_aware_df.index
    )
    
    # Combine with match_id
    hero_features = pd.concat([
        team_aware_df[['match_id']].reset_index(drop=True),
        radiant_features.reset_index(drop=True),
        dire_features.reset_index(drop=True)
    ], axis=1)
    
    print(f"✅ Created {hero_features.shape[1]-1} team-aware hero features")
    
    return hero_features

def analyze_feature_correlations(df, target_col, feature_type="Features"):
    """Analyze correlations between features and target"""
    
    print(f"\n=== {feature_type.upper()} CORRELATION ANALYSIS ===")
    
    target = df[target_col]
    feature_cols = [col for col in df.columns if col not in ['match_id', target_col]]
    
    correlations = []
    for col in feature_cols:
        try:
            # Check if column contains scalar values
            if df[col].dtype in ['float64', 'int64', 'bool', 'uint8']:
                # Additional check for list/array columns
                if not df[col].apply(lambda x: isinstance(x, (list, np.ndarray))).any():
                    corr = df[col].corr(target)
                    if not pd.isna(corr):
                        correlations.append((col, abs(corr), corr))
        except Exception as e:
            print(f"  Skipping column {col}: {e}")
            continue
    
    correlations.sort(key=lambda x: x[1], reverse=True)
    
    print(f"Top 15 strongest correlations:")
    for feature, abs_corr, corr in correlations[:15]:
        print(f"  {feature}: {corr:.4f}")
    
    return correlations

def test_model_performance(X_train, X_test, y_train, y_test, feature_set_name):
    """Test model performance on given features"""
    
    print(f"\n=== TESTING {feature_set_name.upper()} ===")
    print(f"Feature shape: {X_train.shape}")
    
    models = {
        'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
        'XGBoost': xgb.XGBClassifier(random_state=42, eval_metric='logloss', n_estimators=100, verbosity=0)
    }
    
    results = {}
    
    for model_name, model in models.items():
        try:
            # Train model
            model.fit(X_train, y_train)
            
            # Test predictions
            y_pred = model.predict(X_test)
            accuracy = accuracy_score(y_test, y_pred)
            
            # Cross-validation on training data
            cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
            cv_mean = cv_scores.mean()
            cv_std = cv_scores.std()
            
            results[model_name] = {
                'test_accuracy': accuracy,
                'cv_mean': cv_mean,
                'cv_std': cv_std
            }
            
            print(f"{model_name}:")
            print(f"  Test Accuracy: {accuracy:.3f} ({accuracy*100:.1f}%)")
            print(f"  CV Score: {cv_mean:.3f} ± {cv_std:.3f}")
            
        except Exception as e:
            print(f"{model_name}: Error - {e}")
            results[model_name] = None
    
    return results

def main():
    """Main function to run complete model improvement test"""
    
    print("="*60)
    print("COMPLETE MODEL IMPROVEMENT TEST")
    print("="*60)
    
    # Load your data (replace with your actual data loading)
    print("Loading data...")
    
    # Assuming you have these dataframes loaded:
    # main_df = your main dataframe with match_id, radiant_win, player features
    # hero_df = your hero dataframe with match_id, hero_picks
    
    print("Please ensure you have loaded:")
    print("- main_df: DataFrame with match_id, radiant_win, player_hero_*_win_rate columns")  
    print("- hero_df: DataFrame with match_id, hero_picks columns")
    print()
    
    # For demo purposes, let's assume the data is loaded
    # Replace this section with your actual data
    
    return """
    
# USAGE INSTRUCTIONS:
# 1. Load your dataframes:

main_df = your_main_dataframe  # with match_id, radiant_win, player features
hero_df = your_hero_dataframe  # with match_id, hero_picks

# 2. Run the complete test:

def run_complete_test(main_df, hero_df):
    
    print("="*60)
    print("COMPLETE MODEL IMPROVEMENT TEST")
    print("="*60)
    
    # Step 1: Create aggregated player features
    df_with_agg = create_aggregated_player_features(main_df)
    
    # Step 2: Create team-aware hero features
    hero_features = create_team_aware_hero_features(hero_df)
    
    # Step 3: Merge datasets
    print("\\nMerging datasets...")
    combined_df = df_with_agg.merge(hero_features, on='match_id', how='inner')
    print(f"Combined dataset shape: {combined_df.shape}")
    
    # Step 4: Analyze correlations
    target_col = 'radiant_win'
    
    # Analyze aggregated features
    agg_features = ['team_winrate_advantage', 'total_advantage', 'weak_link_advantage', 
                   'star_player_advantage', 'top_players_advantage']
    agg_df = combined_df[['match_id', target_col] + agg_features].copy()
    agg_correlations = analyze_feature_correlations(agg_df, target_col, "Aggregated Features")
    
    # Analyze hero features
    hero_cols = [col for col in combined_df.columns if col.startswith(('radiant_', 'dire_'))]
    hero_df_analysis = combined_df[['match_id', target_col] + hero_cols[:50]].copy()  # Limit to first 50 for speed
    hero_correlations = analyze_feature_correlations(hero_df_analysis, target_col, "Hero Features (Sample)")
    
    # Step 5: Prepare feature sets
    print("\\n" + "="*60)
    print("PREPARING FEATURE SETS FOR TESTING")
    print("="*60)
    
    # Remove match_id and target for ML
    feature_columns = [col for col in combined_df.columns if col not in ['match_id', target_col]]
    X = combined_df[feature_columns]
    y = combined_df[target_col].astype(int)  # Ensure binary target
    
    print(f"Total features available: {len(feature_columns)}")
    print(f"Total samples: {len(X)}")
    print(f"Target distribution: {y.value_counts().values}")
    
    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )
    
    print(f"Train set: {X_train.shape}")
    print(f"Test set: {X_test.shape}")
    
    # Step 6: Test different feature combinations
    print("\\n" + "="*60)
    print("TESTING FEATURE COMBINATIONS")
    print("="*60)
    
    # 1. Aggregated features only
    agg_feature_names = [f for f in agg_features if f in X_train.columns]
    X_train_agg = X_train[agg_feature_names]
    X_test_agg = X_test[agg_feature_names]
    
    results_agg = test_model_performance(X_train_agg, X_test_agg, y_train, y_test, "Aggregated Features Only")
    
    # 2. Hero features only (top 50 by correlation)
    top_hero_features = [feat for feat, _, _ in hero_correlations[:50] if feat in X_train.columns]
    X_train_heroes = X_train[top_hero_features]
    X_test_heroes = X_test[top_hero_features]
    
    results_heroes = test_model_performance(X_train_heroes, X_test_heroes, y_train, y_test, "Hero Features Only")
    
    # 3. Combined features
    combined_features = agg_feature_names + top_hero_features
    X_train_combined = X_train[combined_features]
    X_test_combined = X_test[combined_features]
    
    results_combined = test_model_performance(X_train_combined, X_test_combined, y_train, y_test, "Combined Features")
    
    # Step 7: Performance summary
    print("\\n" + "="*60)
    print("FINAL PERFORMANCE SUMMARY")
    print("="*60)
    
    for model_name in ['Logistic Regression', 'Random Forest', 'XGBoost']:
        print(f"\\n{model_name}:")
        
        try:
            agg_acc = results_agg[model_name]['test_accuracy']
            heroes_acc = results_heroes[model_name]['test_accuracy'] 
            combined_acc = results_combined[model_name]['test_accuracy']
            
            print(f"  Aggregated only:  {agg_acc:.3f} ({agg_acc*100:.1f}%)")
            print(f"  Heroes only:      {heroes_acc:.3f} ({heroes_acc*100:.1f}%)")
            print(f"  Combined:         {combined_acc:.3f} ({combined_acc*100:.1f}%)")
            
            improvement = combined_acc - agg_acc
            total_improvement = combined_acc - 0.537  # vs original baseline
            
            print(f"  Improvement from heroes: +{improvement:.3f} ({improvement*100:.1f}pp)")
            print(f"  Total improvement: +{total_improvement:.3f} ({total_improvement*100:.1f}pp)")
            
            if improvement > 0.01:
                print(f"  ✅ Heroes add significant value!")
            elif improvement > 0.005:
                print(f"  ⚠️  Heroes add modest value")
            else:
                print(f"  ❌ Heroes don't help much")
                
        except:
            print(f"  Error in results for {model_name}")
    
    # Step 8: Feature importance analysis
    print("\\n" + "="*60)
    print("FEATURE IMPORTANCE ANALYSIS")
    print("="*60)
    
    # Train final Random Forest for feature importance
    rf_final = RandomForestClassifier(n_estimators=100, random_state=42)
    rf_final.fit(X_train_combined, y_train)
    
    feature_importance = pd.DataFrame({
        'feature': combined_features,
        'importance': rf_final.feature_importances_
    }).sort_values('importance', ascending=False)
    
    print("Top 20 most important features:")
    print(feature_importance.head(20).to_string(index=False))
    
    # Importance breakdown
    agg_importance = feature_importance[feature_importance['feature'].isin(agg_feature_names)]['importance'].sum()
    hero_importance = feature_importance[~feature_importance['feature'].isin(agg_feature_names)]['importance'].sum()
    
    print(f"\\nImportance breakdown:")
    print(f"  Aggregated features: {agg_importance:.3f} ({agg_importance*100:.1f}%)")
    print(f"  Hero features: {hero_importance:.3f} ({hero_importance*100:.1f}%)")
    
    print("\\n" + "="*60)
    print("TEST COMPLETE!")
    print("="*60)
    
    return {
        'aggregated_results': results_agg,
        'hero_results': results_heroes, 
        'combined_results': results_combined,
        'feature_importance': feature_importance
    }

# Run the test:
# results = run_complete_test(main_df, hero_df)
"""

if __name__ == "__main__":
    main()

COMPLETE MODEL IMPROVEMENT TEST
Loading data...
Please ensure you have loaded:
- main_df: DataFrame with match_id, radiant_win, player_hero_*_win_rate columns
- hero_df: DataFrame with match_id, hero_picks columns



In [70]:
# Debug and fix the data shape issue
def debug_and_fix_data(combined_df):
    """Debug the broadcasting error and fix data shapes"""
    
    print("=== DEBUGGING DATA SHAPES ===")
    
    target_col = 'radiant_win'
    
    # Check target column shape and type
    print(f"Target column '{target_col}':")
    print(f"  Shape: {combined_df[target_col].shape}")
    print(f"  Type: {type(combined_df[target_col].iloc[0])}")
    print(f"  Sample values: {combined_df[target_col].head().tolist()}")
    
    # Fix target if it's not 1D
    if len(combined_df[target_col].shape) > 1:
        print("  🔧 Fixing target shape...")
        combined_df[target_col] = combined_df[target_col].iloc[:, 0]  # Take first column
    
    # Check for problematic columns
    print(f"\nChecking all columns for shape issues...")
    problematic_cols = []
    
    for col in combined_df.columns:
        try:
            # Test if column can be used for correlation
            if col != target_col:
                sample_val = combined_df[col].iloc[0]
                if isinstance(sample_val, (list, np.ndarray)) and len(np.array(sample_val).shape) > 0:
                    problematic_cols.append(col)
                    print(f"  ❌ {col}: contains arrays/lists")
        except:
            problematic_cols.append(col)
            print(f"  ❌ {col}: other issue")
    
    print(f"\nFound {len(problematic_cols)} problematic columns")
    
    # Remove problematic columns
    clean_df = combined_df.drop(columns=problematic_cols)
    
    # Ensure target is boolean/int
    clean_df[target_col] = clean_df[target_col].astype(int)
    
    print(f"Clean dataset shape: {clean_df.shape}")
    print(f"Target distribution: {clean_df[target_col].value_counts().to_dict()}")
    
    return clean_df

# Apply the fix
print("Applying data fix...")
clean_combined_df = debug_and_fix_data(merged_df)

# Now test hero features properly
def test_hero_features_fixed(clean_df):
    """Test hero features with clean data"""
    
    target_col = 'radiant_win'
    
    # Get hero features
    hero_cols = [col for col in clean_df.columns if col.startswith(('radiant_', 'dire_'))]
    print(f"Found {len(hero_cols)} hero features")
    
    if len(hero_cols) > 0:
        # Test correlations on sample
        hero_sample = hero_cols[:50]  # First 50 heroes
        
        print(f"\n=== HERO FEATURES CORRELATION ANALYSIS (FIXED) ===")
        correlations = []
        
        for col in hero_sample:
            try:
                corr = clean_df[col].corr(clean_df[target_col])
                if not pd.isna(corr):
                    correlations.append((col, abs(corr), corr))
            except Exception as e:
                print(f"  Error with {col}: {e}")
        
        correlations.sort(key=lambda x: x[1], reverse=True)
        
        print(f"Top 15 hero correlations:")
        for feature, abs_corr, corr in correlations[:15]:
            print(f"  {feature}: {corr:.4f}")
        
        return correlations
    
    return []

# Test hero features
hero_correlations = test_hero_features_fixed(clean_combined_df)

Applying data fix...
=== DEBUGGING DATA SHAPES ===
Target column 'radiant_win':
  Shape: (96821,)
  Type: <class 'numpy.bool'>
  Sample values: [True, False, True, True, True]

Checking all columns for shape issues...

Found 0 problematic columns
Clean dataset shape: (96821, 15)
Target distribution: {1: 48489, 0: 48332}
Found 4 hero features

=== HERO FEATURES CORRELATION ANALYSIS (FIXED) ===
Top 15 hero correlations:
  radiant_win: 1.0000
  radiant_dire_matchup: 0.1425
  dire_win_rate: -0.1028
  radiant_win_rate: 0.1027


In [71]:
# Run complete test with fixed data
def run_fixed_test(clean_df):
    """Run complete test with fixed data"""
    
    target_col = 'radiant_win'
    
    # Prepare features
    feature_cols = [col for col in clean_df.columns if col not in ['match_id', target_col]]
    X = clean_df[feature_cols]
    y = clean_df[target_col]
    
    print(f"Fixed dataset: {X.shape} features, {len(y)} samples")
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )
    
    # Define feature sets
    agg_features = [col for col in feature_cols if any(keyword in col for keyword in 
                   ['advantage', 'avg', 'min', 'max', 'percentile', 'consistency', 'total'])]
    hero_features = [col for col in feature_cols if col.startswith(('radiant_', 'dire_'))]
    
    print(f"Feature breakdown:")
    print(f"  Aggregated: {len(agg_features)}")
    print(f"  Heroes: {len(hero_features)}")
    
    # Test combinations
    results = {}
    
    # 1. Aggregated only
    if agg_features:
        X_train_agg = X_train[agg_features]
        X_test_agg = X_test[agg_features]
        results['aggregated'] = test_model_performance(X_train_agg, X_test_agg, y_train, y_test, "Aggregated Only")
    
    # 2. Heroes only (top 30)
    if len(hero_correlations) > 0:
        top_heroes = [feat for feat, _, _ in hero_correlations[:30]]
        available_heroes = [col for col in top_heroes if col in X_train.columns]
        
        if available_heroes:
            X_train_heroes = X_train[available_heroes]
            X_test_heroes = X_test[available_heroes]
            results['heroes'] = test_model_performance(X_train_heroes, X_test_heroes, y_train, y_test, "Heroes Only")
    
    # 3. Combined
    combined_features = agg_features + (available_heroes if 'available_heroes' in locals() else [])
    if combined_features:
        X_train_combined = X_train[combined_features]
        X_test_combined = X_test[combined_features]
        results['combined'] = test_model_performance(X_train_combined, X_test_combined, y_train, y_test, "Combined")
    
    # Summary
    print(f"\n=== FINAL RESULTS SUMMARY ===")
    
    for model_name in ['Logistic Regression', 'Random Forest', 'XGBoost']:
        print(f"\n{model_name}:")
        
        baseline = 0.537
        for test_name, test_results in results.items():
            if test_results and model_name in test_results and test_results[model_name]:
                acc = test_results[model_name]['test_accuracy']
                improvement = (acc - baseline) * 100
                print(f"  {test_name.title():12}: {acc:.3f} ({acc*100:.1f}%) [+{improvement:.1f}pp]")
    
    return results

# Run the fixed test
final_results = run_fixed_test(clean_combined_df)

Fixed dataset: (96821, 13) features, 96821 samples
Feature breakdown:
  Aggregated: 0
  Heroes: 3

=== TESTING HEROES ONLY ===
Feature shape: (67774, 3)
Logistic Regression:
  Test Accuracy: 0.570 (57.0%)
  CV Score: 0.565 ± 0.002
Random Forest:
  Test Accuracy: 0.547 (54.7%)
  CV Score: 0.545 ± 0.004
XGBoost:
  Test Accuracy: 0.560 (56.0%)
  CV Score: 0.557 ± 0.003

=== TESTING COMBINED ===
Feature shape: (67774, 3)
Logistic Regression:
  Test Accuracy: 0.570 (57.0%)
  CV Score: 0.565 ± 0.002
Random Forest:
  Test Accuracy: 0.547 (54.7%)
  CV Score: 0.545 ± 0.004
XGBoost:
  Test Accuracy: 0.560 (56.0%)
  CV Score: 0.557 ± 0.003

=== FINAL RESULTS SUMMARY ===

Logistic Regression:
  Heroes      : 0.570 (57.0%) [+3.3pp]
  Combined    : 0.570 (57.0%) [+3.3pp]

Random Forest:
  Heroes      : 0.547 (54.7%) [+1.0pp]
  Combined    : 0.547 (54.7%) [+1.0pp]

XGBoost:
  Heroes      : 0.560 (56.0%) [+2.3pp]
  Combined    : 0.560 (56.0%) [+2.3pp]
